# Adult Income Prediction — Modelling Pipeline
## Work Plan 4: Baseline & Symbolic Models + Evaluation Setup
## Work Plan 6: Advanced Models, Ensemble & Interpretation


In [ ]:
# =============================================================================
# STEP 0: Imports
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.labelsize": 10})


---
## STEP 1 — Load Preprocessed Data
Load the train/val/test splits saved by `EDA_Preprocessing.ipynb`.


In [ ]:
# =============================================================================
# STEP 1 — Load Preprocessed Data
# =============================================================================
# NOTE: You must run EDA_Preprocessing.ipynb first (including the save cell)
X_train = pd.read_csv("data/processed/X_train.csv")
X_val   = pd.read_csv("data/processed/X_val.csv")
X_test  = pd.read_csv("data/processed/X_test.csv")
y_train = pd.read_csv("data/processed/y_train.csv").values.ravel()
y_val   = pd.read_csv("data/processed/y_val.csv").values.ravel()
y_test  = pd.read_csv("data/processed/y_test.csv").values.ravel()

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Train class distribution:")
print(pd.Series(y_train).value_counts(normalize=True).to_string())


---
## STEP 2 — Reusable Evaluation Function
Primary metric: **F1-Score (weighted)**. Secondary: ROC-AUC, Accuracy, Precision, Recall.


In [ ]:
# =============================================================================
# STEP 2 — Evaluation Helper
# =============================================================================
all_results = []  # collect results for final comparison

def evaluate_model(model, X_test, y_test, model_name="Model"):
    """Evaluate a fitted model and print all required metrics."""
    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_proba = model.decision_function(X_test)
    else:
        y_proba = y_pred

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted")
    rec  = recall_score(y_test, y_pred, average="weighted")
    f1   = f1_score(y_test, y_pred, average="weighted")
    auc  = roc_auc_score(y_test, y_proba)

    print(f"\n{'='*60}")
    print(f"  Evaluation: {model_name}")
    print(f"{'='*60}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-Score : {f1:.4f}  << PRIMARY")
    print(f"  ROC-AUC  : {auc:.4f}")
    print(f"\nClassification Report:\n")
    print(classification_report(y_test, y_pred, target_names=["<=50K", ">50K"]))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=["<=50K", ">50K"],
        cmap="Blues", ax=axes[0])
    axes[0].set_title(f"{model_name} - Confusion Matrix")
    RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
    axes[1].set_title(f"{model_name} - ROC Curve")
    plt.tight_layout()
    plt.show()

    result = {"Model": model_name, "Accuracy": acc, "Precision": prec,
              "Recall": rec, "F1-Score": f1, "ROC-AUC": auc}
    all_results.append(result)
    return result


---
# WORK PLAN 4 — Baseline & Symbolic Models
## STEP 3 — Majority-Class Baseline


In [ ]:
# =============================================================================
# STEP 3 — Majority-Class Baseline
# =============================================================================
baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train, y_train)
evaluate_model(baseline, X_test, y_test, "Majority-Class Baseline")


---
## STEP 4 — Logistic Regression + Coefficient Analysis


In [ ]:
# =============================================================================
# STEP 4a — Logistic Regression
# =============================================================================
lr_model = LogisticRegression(max_iter=1000, random_state=42, solver="lbfgs")
lr_model.fit(X_train, y_train)
evaluate_model(lr_model, X_test, y_test, "Logistic Regression")


In [ ]:
# =============================================================================
# STEP 4b — Coefficient Analysis (required by work plan)
# =============================================================================
coeffs = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lr_model.coef_[0]
}).sort_values("Coefficient", ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#DD8452" if c > 0 else "#4C72B0" for c in coeffs["Coefficient"]]
ax.barh(coeffs["Feature"], coeffs["Coefficient"], color=colors, edgecolor="white")
ax.set_title("Logistic Regression - Feature Coefficients", fontsize=14, fontweight="bold")
ax.set_xlabel("Coefficient Value")
ax.axvline(x=0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

print("\nTop 5 features pushing towards >50K:")
print(coeffs.head(5).to_string(index=False))
print("\nTop 5 features pushing towards <=50K:")
print(coeffs.tail(5).to_string(index=False))


---
## STEP 5 — Decision Tree (max_depth=5) + Visualization


In [ ]:
# =============================================================================
# STEP 5a — Decision Tree (max_depth=5 per work plan)
# =============================================================================
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
evaluate_model(dt_model, X_test, y_test, "Decision Tree (depth=5)")


In [ ]:
# =============================================================================
# STEP 5b — Tree Visualization (required by work plan)
# =============================================================================
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(dt_model, feature_names=X_train.columns.tolist(),
          class_names=["<=50K", ">50K"], filled=True, rounded=True,
          fontsize=8, ax=ax)
ax.set_title("Decision Tree Visualization (max_depth=5)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


---
## STEP 6 — Work Plan 4 Results Summary


In [ ]:
# =============================================================================
# STEP 6 — WP4 Results Summary
# =============================================================================
wp4_df = pd.DataFrame(all_results).set_index("Model")
print("\n" + "="*70)
print("  WORK PLAN 4 — MODEL COMPARISON SUMMARY")
print("="*70)
print(wp4_df.round(4).to_string())
print("\nPrimary metric: F1-Score (weighted)")


---
# WORK PLAN 6 — Advanced Models, Ensemble & Interpretation


## STEP 7 — SVM (RBF Kernel)
> Note: SVM can be slow on large datasets. Be patient.


In [ ]:
# =============================================================================
# STEP 7 — SVM (RBF kernel)
# =============================================================================
svm_model = SVC(kernel="rbf", probability=True, random_state=42)
svm_model.fit(X_train, y_train)
evaluate_model(svm_model, X_test, y_test, "SVM (RBF kernel)")


---
## STEP 8 — Random Forest (n_estimators=200)


In [ ]:
# =============================================================================
# STEP 8 — Random Forest (n_estimators=200 per work plan)
# =============================================================================
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
evaluate_model(rf_model, X_test, y_test, "Random Forest (n=200)")


---
## STEP 9 — XGBoost (n_estimators=300)


In [ ]:
# =============================================================================
# STEP 9 — XGBoost (n_estimators=300 per work plan)
# =============================================================================
xgb_model = XGBClassifier(
    n_estimators=300, random_state=42,
    use_label_encoder=False, eval_metric="logloss", n_jobs=-1)
xgb_model.fit(X_train, y_train)
evaluate_model(xgb_model, X_test, y_test, "XGBoost (n=300)")


---
## STEP 10 — Soft Voting Ensemble (LR + RF + XGBoost)


In [ ]:
# =============================================================================
# STEP 10 — Soft Voting Ensemble
# =============================================================================
ensemble_model = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000, random_state=42, solver="lbfgs")),
        ("rf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
        ("xgb", XGBClassifier(n_estimators=300, random_state=42,
                              use_label_encoder=False, eval_metric="logloss", n_jobs=-1))
    ],
    voting="soft"
)
ensemble_model.fit(X_train, y_train)
evaluate_model(ensemble_model, X_test, y_test, "Soft Voting Ensemble (LR+RF+XGB)")


---
## STEP 11 — Feature Importance Analysis (Gini for RF, Gain for XGBoost)


In [ ]:
# =============================================================================
# STEP 11 — Feature Importance
# =============================================================================
rf_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance (Gini)": rf_model.feature_importances_
}).sort_values("Importance (Gini)", ascending=False)

xgb_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance (Gain)": xgb_model.feature_importances_
}).sort_values("Importance (Gain)", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].barh(rf_importance["Feature"], rf_importance["Importance (Gini)"],
             color="#4C72B0", edgecolor="white")
axes[0].set_title("Random Forest - Feature Importance (Gini)", fontsize=13, fontweight="bold")
axes[0].invert_yaxis()

axes[1].barh(xgb_importance["Feature"], xgb_importance["Importance (Gain)"],
             color="#DD8452", edgecolor="white")
axes[1].set_title("XGBoost - Feature Importance (Gain)", fontsize=13, fontweight="bold")
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 5 Features (Random Forest - Gini):")
print(rf_importance.head(5).to_string(index=False))
print("\nTop 5 Features (XGBoost - Gain):")
print(xgb_importance.head(5).to_string(index=False))


---
## STEP 12 — Decision Tree Rule Extraction


In [ ]:
# =============================================================================
# STEP 12 — Decision Tree Rules
# =============================================================================
tree_rules = export_text(dt_model, feature_names=X_train.columns.tolist())
print("Decision Tree Rules (max_depth=5):")
print(tree_rules)


---
## STEP 13 — Fairness Analysis Across Protected Attributes
Analyzing model performance across `sex`, `race` groups.


In [ ]:
# =============================================================================
# STEP 13 — Fairness Analysis
# =============================================================================
# We analyze fairness using the encoded features available in X_test.
# sex_Male=True means Male, sex_Male=False means Female
# race columns: race_White, race_Black, race_Asian-Pac-Islander, race_Other

def fairness_analysis(model, X_test_df, y_test_arr, group_col, model_name):
    """Analyze model fairness across a binary group column."""
    y_pred = model.predict(X_test_df)
    results = []
    for val in [True, False]:
        mask = X_test_df[group_col] == val
        if mask.sum() < 10:
            continue
        g_f1 = f1_score(y_test_arr[mask], y_pred[mask], average="weighted", zero_division=0)
        g_acc = accuracy_score(y_test_arr[mask], y_pred[mask])
        results.append({"Group": f"{group_col}={val}", "Size": mask.sum(),
                        "Accuracy": round(g_acc, 4), "F1-Score": round(g_f1, 4)})
    df = pd.DataFrame(results)
    print(f"\nFairness: {model_name} by {group_col}")
    print(df.to_string(index=False))
    return df

# Analyze fairness for the best model (Ensemble)
best_model = ensemble_model
best_name = "Soft Voting Ensemble"

# Check which columns exist in X_test
fair_cols = [c for c in X_test.columns if c.startswith("sex_") or c.startswith("race_")]
print(f"Protected attribute columns found: {fair_cols}")

for col in fair_cols:
    fairness_analysis(best_model, X_test, y_test, col, best_name)


---
## STEP 14 — Final Combined Model Comparison (WP4 + WP6)


In [ ]:
# =============================================================================
# STEP 14 — Final Combined Comparison
# =============================================================================
final_df = pd.DataFrame(all_results).set_index("Model")
print("\n" + "="*70)
print("  COMPLETE MODEL COMPARISON (Work Plans 4 + 6)")
print("="*70)
print(final_df.round(4).to_string())
print("\nPrimary metric: F1-Score (weighted)")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(12, 6))
final_df[["F1-Score", "ROC-AUC"]].plot(kind="bar", ax=ax, edgecolor="white")
ax.set_title("Model Comparison: F1-Score & ROC-AUC", fontsize=14, fontweight="bold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
